# Evidence Detection (ED) — DeBERTa v3 5-fold ensemble
Binary classification for (claim, evidence) pairs using a 5-fold DeBERTa v3 ensemble with averaged probabilities and tuned threshold.


## Install and Imports


In [ ]:
import os, re, random, json
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score

# for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


## Model Training


### Config and load data


In [ ]:
CONFIG = {
    "model_name": "microsoft/deberta-v3-base",
    "max_length": 256,
    "train_bs": 16,
    "eval_bs": 32,
    "lr": 2e-5,
    "epochs": 3,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "n_folds": 5,
    "out_dir": "./outputs/deberta_v3_5_fold_ensemble",
    "ensemble_info_path": "./outputs/deberta_v3_5_fold_ensemble/ensemble_info.json",
    "train_path": "./training_data/train.csv",
    "dev_path": "./training_data/dev.csv",
}

TEST_MODE = False  # set True for quick testing, False for full training

# load data into dataframes
train_df = pd.read_csv(CONFIG["train_path"])
dev_df   = pd.read_csv(CONFIG["dev_path"])

# ensure labels are integers
train_df["label"] = train_df["label"].astype(int)
dev_df["label"]   = dev_df["label"].astype(int)

# combine labelled rows so each sample can appear in a held-out fold once
full_df = pd.concat([train_df, dev_df], ignore_index=True)

if TEST_MODE:
    full_df = full_df.sample(256, random_state=SEED).reset_index(drop=True)

# print dataset stats
print("Train:", len(train_df), "Dev:", len(dev_df), "Full:", len(full_df))
print("Full label dist:", full_df["label"].value_counts().to_dict())
full_df.head(3)


### Tokenisation


In [ ]:
# create dataset and tokeniser
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], use_fast=True)
full_ds = Dataset.from_pandas(full_df[["Claim", "Evidence", "label"]], preserve_index=False)

# tokenize the dataset
def tok_fn(batch):
    return tokenizer(
        batch["Claim"],
        batch["Evidence"],
        truncation=True,
        max_length=CONFIG["max_length"],
    )

full_tok = full_ds.map(tok_fn, batched=True, remove_columns=["Claim", "Evidence"])

# pads inputs to the same length in a batch
collator = DataCollatorWithPadding(tokenizer=tokenizer)
labels = full_df["label"].to_numpy()


### Training and metrics


In [ ]:
def logits_to_pos_probs(logits: np.ndarray) -> np.ndarray:
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_vals = np.exp(shifted)
    probs = exp_vals / np.sum(exp_vals, axis=1, keepdims=True)
    return probs[:, 1]


def binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}


def tune_threshold(y_true: np.ndarray, probs: np.ndarray) -> tuple[float, float]:
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 181):
        preds = (probs >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = float(t), float(f1)
    return best_t, best_f1


def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    probs = logits_to_pos_probs(logits)
    preds = (probs >= 0.5).astype(int)
    return binary_metrics(y_true, preds)


skf = StratifiedKFold(n_splits=CONFIG["n_folds"], shuffle=True, random_state=SEED)
oof_probs = np.zeros(len(full_df), dtype=np.float32)
fold_model_dirs = []

for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), start=1):
    fold_dir = os.path.join(CONFIG["out_dir"], f"fold_{fold_idx}")
    best_dir = os.path.join(fold_dir, "best_model")
    os.makedirs(fold_dir, exist_ok=True)

    # load pretrained DeBERTa v3 with a classification head for 2 labels
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["model_name"],
        num_labels=2,
        id2label={0: "not_relevant", 1: "relevant"},
        label2id={"not_relevant": 0, "relevant": 1},
    )

    args = TrainingArguments(
        output_dir=fold_dir,
        learning_rate=CONFIG["lr"],
        per_device_train_batch_size=CONFIG["train_bs"],
        per_device_eval_batch_size=CONFIG["eval_bs"],
        max_steps=2 if TEST_MODE else -1,
        num_train_epochs=CONFIG["epochs"] if not TEST_MODE else 1,
        weight_decay=CONFIG["weight_decay"],
        warmup_ratio=CONFIG["warmup_ratio"] if not TEST_MODE else 0.0,
        eval_strategy="epoch",
        save_strategy="epoch" if not TEST_MODE else "steps",
        save_steps=1 if TEST_MODE else None,
        load_best_model_at_end=True if not TEST_MODE else False,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=1,
        fp16=torch.cuda.is_available() if not TEST_MODE else False,
        report_to="none",
        seed=SEED + fold_idx,
        data_seed=SEED + fold_idx,
        disable_tqdm=True,
    )

    train_fold = full_tok.select(tr_idx.tolist())
    val_fold = full_tok.select(val_idx.tolist())

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_fold,
        eval_dataset=val_fold,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # gather held-out fold probabilities for threshold tuning
    val_pred = trainer.predict(val_fold)
    val_probs = logits_to_pos_probs(val_pred.predictions)
    oof_probs[val_idx] = val_probs

    # save fold model + tokenizer
    trainer.save_model(best_dir)
    tokenizer.save_pretrained(best_dir)
    fold_model_dirs.append(best_dir)

    fold_preds = (val_probs >= 0.5).astype(int)
    fold_stats = binary_metrics(labels[val_idx], fold_preds)
    print(f"Fold {fold_idx} @0.5 metrics:", fold_stats)

best_threshold, best_oof_f1 = tune_threshold(labels, oof_probs)
oof_preds = (oof_probs >= best_threshold).astype(int)
oof_stats = binary_metrics(labels, oof_preds)

print(f"Tuned threshold (OOF): {best_threshold:.3f}  |  F1={best_oof_f1:.4f}")
print("OOF metrics with tuned threshold:", oof_stats)


### Save model


In [ ]:
# Save ensemble metadata (all fold models are already saved in-fold)
os.makedirs(CONFIG["out_dir"], exist_ok=True)
ensemble_info = {
    "model_name": CONFIG["model_name"],
    "strategy": "5_fold",
    "model_dirs": fold_model_dirs,
    "threshold": float(best_threshold),
    "max_length": CONFIG["max_length"],
}

with open(CONFIG["ensemble_info_path"], "w") as f:
    json.dump(ensemble_info, f, indent=2)

print("Saved ensemble metadata to:", CONFIG["ensemble_info_path"])
ensemble_info


## Model Demo


In [ ]:
# specify test file path and output path for predictions
TEST_PATH = "./test.csv"
OUT_PATH  = "./predictions_5_fold_ensemble.csv"


@torch.inference_mode()  # disables gradient tracking to make inference faster/cheaper
def predict_test_ensemble(model_dirs: list[str], threshold: float, test_csv: str, out_csv: str) -> pd.DataFrame:
    # load tokenizer from first fold model
    tok = AutoTokenizer.from_pretrained(model_dirs[0], use_fast=True)

    # load test file
    df = pd.read_csv(test_csv)

    # convert to HF dataset
    ds = Dataset.from_pandas(df[["Claim", "Evidence"]], preserve_index=False)

    # tokenise pairs in same way as training
    def tok_fn(batch):
        return tok(batch["Claim"], batch["Evidence"], truncation=True, max_length=CONFIG["max_length"])

    ds_tok = ds.map(tok_fn, batched=True, remove_columns=["Claim", "Evidence"])

    # pad batches
    data_collator = DataCollatorWithPadding(tokenizer=tok)

    # average probabilities from all fold models
    all_probs = []
    for model_dir in model_dirs:
        mdl = AutoModelForSequenceClassification.from_pretrained(model_dir)
        infer_args = TrainingArguments(
            output_dir=os.path.join(model_dir, "_infer_tmp"),
            per_device_eval_batch_size=64,
            report_to="none",
            disable_tqdm=True,
        )
        infer_trainer = Trainer(model=mdl, args=infer_args, processing_class=tok, data_collator=data_collator)
        pred = infer_trainer.predict(ds_tok)
        probs = logits_to_pos_probs(pred.predictions)
        all_probs.append(probs)

    mean_probs = np.mean(np.vstack(all_probs), axis=0)
    yhat = (mean_probs >= threshold).astype(int)

    # write output CSV
    out = df.copy()
    out["prob_relevant"] = mean_probs
    out["pred"] = yhat
    out.to_csv(out_csv, index=False)

    return out


# Run demo inference and preview results
demo = predict_test_ensemble(fold_model_dirs, best_threshold, TEST_PATH, OUT_PATH)
demo.head(5), print("Wrote:", OUT_PATH)
